# <font color="#418FDE" size="6.5" uppercase>**Verluste und Metriken**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Berechnen Regressionsverluste aus tatsächlichen Werten und festen Vorhersagen. 
- Erstellen einfache Baselines ohne Training und vergleichen sie mit Beispielvorhersagen. 
- Berechnen Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen manuell. 


## **1. Regressionsverluste verstehen**

### **1.1. Werte und Vorhersagen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_01.jpg?v=1784789220" width="250">



>* Regression sagt vergleichbare Zahlenwerte voraus
>* Reale Werte und Prognosen paarweise vergleichen

>* Gleiche Einheit und Bedeutung sicherstellen
>* Zielgröße, Zeitraum und Vergleichbarkeit prüfen

>* Vorhersagen liegen bereits ohne Training vor
>* Vergleiche Realität und Prognose pro Fall



In [ ]:
#@title Python-Code - Werte und Vorhersagen

# Wir vergleichen echte Werte mit festen Vorhersagen.
# Kleine Abweichungen werden als Fehler sichtbar.
# Am Ende sehen wir einfache Regressionsverluste.

import numpy as np
import pandas as pd

# Diese Werte stehen paarweise in derselben Einheit.
actual_kwh = np.array([18, 22, 20, 25, 19], dtype=float)
predicted_kwh = np.array([17, 24, 21, 23, 20], dtype=float)

# Eine kurze Prüfung verhindert unpassende Vergleiche.
if actual_kwh.shape != predicted_kwh.shape:
    raise ValueError("Tatsächliche Werte und Vorhersagen brauchen gleiche Länge.")

# Der Fehler zeigt Richtung und Größe der Abweichung.
error_kwh = predicted_kwh - actual_kwh
absolute_error_kwh = np.abs(error_kwh)

# Quadratische Fehler bestrafen größere Abweichungen stärker.
squared_error_kwh2 = error_kwh ** 2

# Diese Mittelwerte fassen alle Einzelfehler zusammen.
mae_kwh = np.mean(absolute_error_kwh)
mse_kwh2 = np.mean(squared_error_kwh2)
rmse_kwh = np.sqrt(mse_kwh2)

# Die Tabelle zeigt die paarweisen Vergleiche kompakt.
comparison = pd.DataFrame(
    {
        "Ist_kWh": actual_kwh,
        "Vorhersage_kWh": predicted_kwh,
        "Fehler_kWh": error_kwh,
        "Absolut_kWh": absolute_error_kwh,
        "Quadrat_kWh2": squared_error_kwh2,
    }
)

print("Paarweiser Vergleich von Ist-Werten und festen Vorhersagen:")
print(comparison.round(2).to_string(index=False))
print(f"MAE: {mae_kwh:.2f} kWh")
print(f"MSE: {mse_kwh2:.2f} kWh²")
print(f"RMSE: {rmse_kwh:.2f} kWh")



### **1.2. Fehlerarten erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_02.jpg?v=1784789222" width="250">



>* Fehler zeigen Unter- oder Überschätzung
>* Die Richtung hat praktische Folgen

>* Fehlergröße immer im Kontext bewerten
>* Verlustwerte hängen von Fehlerarten ab

>* Systematische Fehler zeigen wiederkehrende Abweichungsmuster
>* Ausreißer beeinflussen Verlustwerte besonders stark



In [ ]:
#@title Python-Code - Fehlerarten erkennen

# Wir erkennen Fehlerarten bei festen Regressionsvorhersagen.
# Richtung und Größe werden getrennt betrachtet.
# Die Ausgabe zeigt typische Fehlermuster.

import numpy as np
import pandas as pd

# Kleine Beispieldaten halten die Rechnung übersichtlich.
actual = np.array([100, 120, 90, 150, 130], dtype=float)
predicted = np.array([95, 135, 90, 170, 110], dtype=float)

# Gleiche Länge ist für paarweise Fehler notwendig.
if actual.shape != predicted.shape:
    raise ValueError("Tatsächliche Werte und Vorhersagen müssen gleich lang sein.")

# Der vorzeichenbehaftete Fehler zeigt die Richtung.
error = predicted - actual
absolute_error = np.abs(error)

# Einfache Textlabels machen die Fehlerarten sichtbar.
error_type = np.where(error < 0, "Unterschätzung", "Überschätzung")
error_type = np.where(error == 0, "Treffer", error_type)

# Eine kleine Tabelle verbindet Werte, Fehler und Bedeutung.
result = pd.DataFrame(
    {
        "Ist": actual,
        "Vorhersage": predicted,
        "Fehler": error,
        "Art": error_type,
        "Betrag": absolute_error,
    }
)

# Zusammenfassungen zeigen systematische Richtung und typische Größe.
mean_error = error.mean()
mean_absolute_error = absolute_error.mean()

print("Fehlerarten für feste Vorhersagen:")
print(result.to_string(index=False))
print(f"Mittlerer Fehler: {mean_error:.1f} Einheiten")
print(f"Mittlerer absoluter Fehler: {mean_absolute_error:.1f} Einheiten")
print("Positiv bedeutet im Mittel Überschätzung, negativ Unterschätzung.")



### **1.3. Verluste berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_01_03.jpg?v=1784789224" width="250">



>* Verlust fasst Vorhersagefehler zusammen
>* Kleinerer Verlust bedeutet genauere Vorhersagen

>* Absolute Fehler zeigen durchschnittliche Abweichungen.
>* Große Fehler können stärker zählen.

>* Verluste Schritt für Schritt berechnen
>* Verlustwerte immer im Kontext bewerten



In [ ]:
#@title Python-Code - Verluste berechnen

# Dieses Beispiel berechnet Verluste aus festen Vorhersagen.
# Absolute und quadratische Fehler werden direkt verglichen.
# Kleinere Werte bedeuten hier genauere Preisvorhersagen.

import numpy as np
import pandas as pd

# Tatsächliche und vorhergesagte Mietpreise stehen paarweise zusammen.
actual_prices = np.array([820, 950, 1100, 760, 1250], dtype=float)
predicted_prices = np.array([800, 1000, 1040, 790, 1400], dtype=float)

# Diese Prüfung verhindert unpassende Längen beim Vergleichen.
if actual_prices.shape != predicted_prices.shape:
    raise ValueError("Tatsächliche Werte und Vorhersagen müssen gleich lang sein.")

# Fehler behalten die Richtung der Abweichung.
errors = predicted_prices - actual_prices
absolute_errors = np.abs(errors)
squared_errors = errors ** 2

# MAE mittelt absolute Fehler, MSE mittelt quadrierte Fehler.
mae = np.mean(absolute_errors)
mse = np.mean(squared_errors)
rmse = np.sqrt(mse)

# Eine kleine Tabelle zeigt die einzelnen Beiträge zum Verlust.
results = pd.DataFrame({
    "Ist": actual_prices,
    "Vorhersage": predicted_prices,
    "Fehler": errors,
    "Absolut": absolute_errors,
    "Quadrat": squared_errors,
})

# Gerundete Werte sind für den Einstieg leichter lesbar.
print("Einzelfehler in Euro:")
print(results.round(0).to_string(index=False))
print(f"MAE: {mae:.1f} Euro")
print(f"MSE: {mse:.1f} Euro²")
print(f"RMSE: {rmse:.1f} Euro")



## **2. Einfache Baselines**

### **2.1. Mittelwert als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_01.jpg?v=1784789214" width="250">



>* Mittelwert sagt konstant den Trainingsdurchschnitt voraus
>* Komplexe Modelle sollten diese Basis übertreffen

>* Mittelwert-Baseline ordnet Modellqualität realistisch ein
>* Schlechtere Modelle zeigen mögliche Probleme

>* Mittelwert nur aus Trainingsdaten berechnen
>* Einfach, aber empfindlich gegenüber Ausreißern



In [ ]:
#@title Python-Code - Mittelwert als Baseline

# Wir vergleichen eine Mittelwert-Baseline mit Beispielvorhersagen.
# Die Baseline nutzt nur Trainingszielwerte.
# Die Fehler zeigen den fairen Vergleich.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Trainingsdaten mit Mietpreisen in Euro.
train_rent_eur = np.array([620, 700, 760, 820, 900, 980], dtype=float)
test_rent_eur = np.array([650, 780, 880, 1050], dtype=float)

# Die Baseline sagt immer den Trainingsmittelwert voraus.
baseline_value = train_rent_eur.mean()
baseline_predictions = np.full(test_rent_eur.shape, baseline_value)

# Beispielvorhersagen stehen für ein anderes Modell.
model_predictions = np.array([690, 800, 860, 990], dtype=float)

# Diese Prüfung verhindert unfaire Vergleiche mit falschen Längen.
if len(test_rent_eur) != len(model_predictions):
    raise ValueError("Testwerte und Vorhersagen müssen gleich lang sein.")

# Der mittlere absolute Fehler ist leicht interpretierbar.
baseline_mae = np.mean(np.abs(test_rent_eur - baseline_predictions))
model_mae = np.mean(np.abs(test_rent_eur - model_predictions))

print(f"Trainingsmittelwert: {baseline_value:.0f} Euro")
print(f"MAE der Mittelwert-Baseline: {baseline_mae:.1f} Euro")
print(f"MAE der Beispielvorhersagen: {model_mae:.1f} Euro")

# Die Grafik zeigt echte Werte und beide Vorhersagestrategien.
case_numbers = np.arange(1, len(test_rent_eur) + 1)
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(case_numbers, test_rent_eur, marker="o", label="Tatsächliche Miete")
ax.plot(case_numbers, baseline_predictions, marker="o", label="Mittelwert-Baseline")
ax.plot(case_numbers, model_predictions, marker="o", label="Beispielvorhersagen")

ax.set_title("Mittelwert-Baseline im Vergleich")
ax.set_xlabel("Testfall")
ax.set_ylabel("Miete in Euro")
ax.legend()

plt.show()



### **2.2. Median als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_02.jpg?v=1784789216" width="250">



>* Median-Baseline sagt stets den mittleren Rangwert voraus
>* Robust gegenüber Ausreißern, typischer als Mittelwert

>* Median hilft bei Ausreißern und schiefen Daten
>* Starke Modelle müssen ihn klar übertreffen

>* Median-Baseline als Mindeststandard verstehen
>* Modelle müssen sie klar übertreffen



In [ ]:
#@title Python-Code - Median als Baseline

# Wir vergleichen Median-Baseline und Beispielvorhersagen.
# Der Median ignoriert Merkmale und Ausreißer.
# Die Fehler zeigen den Nutzen der Baseline.

import numpy as np
import pandas as pd

# Kleine Zielwerte simulieren schiefe monatliche Kosten.
actual_costs = np.array([42, 45, 47, 50, 52, 55, 58, 60, 65, 220])

# Diese Beispielvorhersagen wirken plausibel, sind aber nicht trainiert.
example_predictions = np.array([45, 48, 50, 52, 54, 56, 60, 62, 70, 120])

# Die Median-Baseline sagt für alle Fälle denselben Wert voraus.
median_value = np.median(actual_costs)
median_predictions = np.full(actual_costs.shape, median_value)

# Der mittlere absolute Fehler passt gut zur Median-Idee.
median_mae = np.mean(np.abs(actual_costs - median_predictions))
example_mae = np.mean(np.abs(actual_costs - example_predictions))

# Eine kurze Tabelle zeigt echte Werte und beide Vorhersagen.
comparison = pd.DataFrame({
    "Ist-Kosten": actual_costs,
    "Median-Baseline": median_predictions,
    "Beispielvorhersage": example_predictions,
})

# Die Ausgabe bleibt klein und direkt vergleichbar.
print("Median-Baseline:", round(median_value, 1), "Euro")
print("MAE Median-Baseline:", round(median_mae, 1), "Euro")
print("MAE Beispielvorhersage:", round(example_mae, 1), "Euro")
print(comparison.head(5).to_string(index=False))



### **2.3. Mehrheitsklasse als Baseline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_02_03.jpg?v=1784789218" width="250">



>* Sagt immer die häufigste Klasse voraus
>* Dient als Mindestmaßstab für Modellleistung

>* Genauigkeit kann bei seltenen Klassen täuschen
>* Baseline im Anwendungskontext kritisch vergleichen

>* Baseline fair auf Trainingsdaten bestimmen
>* Modell soll seltene Klassen besser erkennen



In [ ]:
#@title Python-Code - Mehrheitsklasse als Baseline

# Diese Übung zeigt eine einfache Klassifikationsbaseline.
# Die Mehrheitsklasse wird ohne Training vorhergesagt.
# Wir vergleichen sie mit Beispielvorhersagen.

import numpy as np
from sklearn.metrics import accuracy_score
import sklearn

# Diese Zielwerte sind absichtlich unausgewogen gewählt.
true_labels = np.array([
    "gesund", "gesund", "gesund", "gesund", "gesund",
    "gesund", "gesund", "krank", "gesund", "krank",
])

# Die häufigste Klasse wird direkt aus den Zielwerten bestimmt.
unique_labels, counts = np.unique(true_labels, return_counts=True)
majority_label = unique_labels[np.argmax(counts)]

# Die Baseline sagt für jede Person dieselbe Klasse voraus.
baseline_predictions = np.full(true_labels.shape, majority_label)

# Ein Beispielmodell erkennt einige seltene Fälle besser.
model_predictions = np.array([
    "gesund", "gesund", "gesund", "krank", "gesund",
    "gesund", "gesund", "krank", "gesund", "krank",
])

# Beide Vorhersagen werden mit derselben Metrik verglichen.
baseline_accuracy = accuracy_score(true_labels, baseline_predictions)
model_accuracy = accuracy_score(true_labels, model_predictions)

# Die Ausgabe bleibt kurz und zeigt den Baseline-Vergleich.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Mehrheitsklasse: {majority_label}")
print(f"Anteil der Mehrheitsklasse: {counts.max() / len(true_labels):.0%}")
print(f"Genauigkeit der Baseline: {baseline_accuracy:.0%}")
print(f"Genauigkeit der Beispielvorhersagen: {model_accuracy:.0%}")
print("Merke: Ein Modell sollte diese einfache Baseline sinnvoll übertreffen.")



## **3. Klassifikationsmetriken manuell berechnen**

### **3.1. Labels und Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_01.jpg?v=1784789209" width="250">



>* Labels zeigen die tatsächliche Klasse.
>* Positive Klasse vor Metrikberechnung festlegen.

>* Scores zeigen Modell-Einschätzungen, keine Entscheidungen
>* Schwellenwerte machen daraus positive oder negative Klassen

>* Schwellenwerte verändern Vorhersagen und Metriken
>* Scores, Labels und Entscheidungen gemeinsam prüfen



In [ ]:
#@title Python-Code - Labels und Scores

# Wir trennen wahre Labels von Modell-Scores.
# Ein Schwellenwert macht daraus Klassenentscheidungen.
# Manuelle Zählungen ergeben einfache Klassifikationsmetriken.

import numpy as np

# Diese kleinen Beispieldaten bleiben vollständig überschaubar.
true_labels = np.array([1, 0, 1, 0, 1, 0, 0, 1])
scores = np.array([0.90, 0.70, 0.65, 0.55, 0.40, 0.35, 0.20, 0.10])

# Die Arrays müssen gleich lang sein.
if len(true_labels) != len(scores):
    raise ValueError("Labels und Scores müssen gleich lang sein.")

# Der Schwellenwert verwandelt Scores in vorhergesagte Labels.
threshold = 0.50
predicted_labels = (scores >= threshold).astype(int)

# Jede Zählung vergleicht wahres Label und Vorhersage.
true_positive = int(np.sum((true_labels == 1) & (predicted_labels == 1)))
false_positive = int(np.sum((true_labels == 0) & (predicted_labels == 1)))
true_negative = int(np.sum((true_labels == 0) & (predicted_labels == 0)))
false_negative = int(np.sum((true_labels == 1) & (predicted_labels == 0)))

# Diese Nenner schützen vor Division durch null.
total = len(true_labels)
positive_predictions = true_positive + false_positive
actual_positives = true_positive + false_negative

# Die Metriken werden direkt aus den Zählungen berechnet.
accuracy = (true_positive + true_negative) / total
precision = true_positive / positive_predictions
recall = true_positive / actual_positives

print("Labels:       " + str(true_labels.tolist()))
print("Scores:       " + str(scores.tolist()))
print("Schwelle:     " + str(threshold))
print("Vorhersagen:  " + str(predicted_labels.tolist()))
print("TP FP TN FN:  " + str([true_positive, false_positive, true_negative, false_negative]))
print("Accuracy:     " + str(round(accuracy, 2)))
print("Precision:    " + str(round(precision, 2)))
print("Recall:       " + str(round(recall, 2)))



### **3.2. Konfusionsmatrix verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_02.jpg?v=1784789210" width="250">



>* Vergleicht wahre und vorhergesagte Klassen
>* Zeigt Fehlerarten statt nur Trefferquote

>* Wahre Labels und Vorhersagen systematisch zählen.
>* Fehlerarten zeigen Warnungen oder übersehene Fälle.

>* Grundlage für Metriken und Kontextbewertung
>* Schwellenwerte zeigen Treffer-Fehlalarm-Zielkonflikte



In [ ]:
#@title Python-Code - Konfusionsmatrix verstehen

# Wir zählen Klassifikationsfehler Schritt für Schritt.
# Die Konfusionsmatrix zeigt vier wichtige Fälle.
# Am Ende vergleichen wir manuelle Kennzahlen.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten halten das Zählen übersichtlich.
actual = np.array([1, 0, 1, 1, 0, 0, 1, 0])
predicted = np.array([1, 0, 0, 1, 1, 0, 1, 0])

# Diese Prüfung verhindert unpassende Listenlängen.
if actual.shape != predicted.shape:
    raise ValueError("Wahre und vorhergesagte Labels müssen gleich lang sein.")

# Jede Zelle wird direkt aus den Labels gezählt.
true_positive = int(np.sum((actual == 1) & (predicted == 1)))
true_negative = int(np.sum((actual == 0) & (predicted == 0)))

false_positive = int(np.sum((actual == 0) & (predicted == 1)))
false_negative = int(np.sum((actual == 1) & (predicted == 0)))

# Die Matrix nutzt Zeilen für Wahrheit und Spalten für Vorhersage.
confusion_matrix = np.array(
    [[true_negative, false_positive], [false_negative, true_positive]]
)

# Aus denselben vier Zahlen entstehen einfache Metriken.
total = len(actual)
accuracy = (true_positive + true_negative) / total

precision = true_positive / (true_positive + false_positive)
recall = true_positive / (true_positive + false_negative)

print("Konfusionsmatrix: Zeilen=wahr 0/1, Spalten=vorhergesagt 0/1")
print(confusion_matrix)
print(f"TN={true_negative}, FP={false_positive}, FN={false_negative}, TP={true_positive}")
print(f"Genauigkeit={accuracy:.2f}, Präzision={precision:.2f}, Sensitivität={recall:.2f}")

# Die Grafik macht die vier Zählfelder sichtbar.
fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(confusion_matrix, cmap="Blues")

ax.set_title("Konfusionsmatrix manuell gezählt")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Tatsächliche Klasse")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["0 negativ", "1 positiv"])
ax.set_yticklabels(["0 negativ", "1 positiv"])

# Jede Zelle erhält ihren Namen und ihre Anzahl.
labels = np.array([["TN", "FP"], ["FN", "TP"]])
for row in range(2):
    for col in range(2):
        text = f"{labels[row, col]}\n{confusion_matrix[row, col]}"
        ax.text(col, row, text, ha="center", va="center", color="black")

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()



### **3.3. Passende Metriken wählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_B/image_03_03.jpg?v=1784789212" width="250">



>* Metriken müssen zum Fehlerkontext passen
>* Genauigkeit täuscht bei ungleichen Klassen

>* Präzision bewertet Verlässlichkeit positiver Vorhersagen
>* Sensitivität findet Positive, Schwellenwerte steuern Kompromisse

>* Schwellenwerte passend zum Anwendungskontext wählen
>* Wahrscheinlichkeiten und Fehlerkosten gemeinsam bewerten



In [ ]:
#@title Python-Code - Passende Metriken wählen

# Wir wählen Metriken passend zum Fehlerziel.
# Schwellenwerte verändern Präzision und Sensitivität sichtbar.
# Die Ausgabe vergleicht drei einfache Entscheidungen.

import numpy as np
import pandas as pd

# Diese festen Werte ersetzen ein trainiertes Modell.
actual = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1])
scores = np.array([0.95, 0.80, 0.70, 0.60, 0.55, 0.45, 0.35, 0.30, 0.20, 0.10])

# Die Prüfung verhindert unklare Metriken bei falschen Längen.
if len(actual) != len(scores):
    raise ValueError("actual und scores müssen gleich lang sein.")

# Drei Schwellenwerte zeigen unterschiedliche fachliche Prioritäten.
thresholds = [0.30, 0.50, 0.70]
rows = []

# Jede Zeile berechnet die wichtigsten Zählungen manuell.
for threshold in thresholds:
    predicted = (scores >= threshold).astype(int)
    tp = int(np.sum((actual == 1) & (predicted == 1)))
    fp = int(np.sum((actual == 0) & (predicted == 1)))
    fn = int(np.sum((actual == 1) & (predicted == 0)))
    tn = int(np.sum((actual == 0) & (predicted == 0)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = (tp + tn) / len(actual)
    rows.append([threshold, accuracy, precision, recall, fp])

# Die Tabelle verbindet Metriken mit einer möglichen Entscheidung.
result = pd.DataFrame(
    rows,
    columns=["Schwelle", "Genauigkeit", "Präzision", "Sensitivität", "Falsch positiv"],
)

# Gerundete Werte sind für den ersten Vergleich leichter lesbar.
print("Metrikwahl: niedrige Schwelle findet mehr Positive, erzeugt aber Alarme.")
print(result.round(2).to_string(index=False))
print("Wenn übersehene Positive teuer sind, zählt Sensitivität besonders.")
print("Wenn falsche Alarme teuer sind, zählt Präzision besonders.")



# <font color="#418FDE" size="6.5" uppercase>**Verluste und Metriken**</font>


In this lecture, you learned to:
- Berechnen Regressionsverluste aus tatsächlichen Werten und festen Vorhersagen. 
- Erstellen einfache Baselines ohne Training und vergleichen sie mit Beispielvorhersagen. 
- Berechnen Klassifikationsmetriken, Schwellenwerte und probabilistische Kennzahlen manuell. 

In the next Module (Module 7), we will go over 'Lineare Regression'